```{admonition} Adapted from
:class: tip
**Source:** The outline of this lab (droplet capture, cell barcodes and UMIs, 3' tag counting, experimental design and the analysis workflow) follows the "Introduction to scRNA-seq" and "Experimental design" lessons of the Harvard Chan Bioinformatics Core (HBC) course *Introduction to single-cell RNA-seq*: Piper M, Mistry M, Liu J, Gammerdinger W, Khetani R (2022), [hbctraining/scRNA-seq_online](https://hbctraining.github.io/Intro-to-scRNAseq/), Zenodo [doi:10.5281/zenodo.5826256](https://doi.org/10.5281/zenodo.5826256). Plant-specific framing (protoplasting, root cell types) draws on the Galaxy Training Network tutorial [*Analysis of plant scRNA-Seq Data with Scanpy*](https://training.galaxyproject.org/training-material/topics/single-cell/tutorials/scrna-plant/tutorial.html) (Tekman M, Serrano-Solano B, Gallardo C, Videm P).

**License:** both sources are [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

**Changes:** The text is re-written for this course and applied to the *Arabidopsis thaliana* root. The plant examples, the dataset discussion and the questions are original.
```

# M8_L1 - Single-Cell RNA-Seq Overview

## From bulk RNA-Seq to single-cell RNA-Seq
In Modules 3 to 6, every RNA-Seq library came from a piece of tissue (a whole root, or a whole shoot). The counts in those libraries are an **average over thousands of cells**, and a gene that is switched on in a single cell layer looks the same as a gene that is weakly expressed everywhere. Single-cell RNA-Seq (scRNA-Seq) keeps track of which cell every transcript came from, so we can ask which cell types are present, which genes define them, and how cells change as they develop.

<u>**By the end of this notebook, you should be familiar with the following concepts:**</u>
- Why single-cell resolution matters, and why the *Arabidopsis* root is a useful model for it
- How plant cells are prepared for scRNA-Seq (protoplasts and nuclei), and the artifacts this can introduce
- The main scRNA-Seq platforms (plate-based, droplet-based, split-pool) and their trade-offs
- How a 10x Genomics Chromium 3' library is built: GEMs, cell barcodes, UMIs, poly(dT) capture and the final Illumina library
- What each read (Read 1, Index 1, Read 2) in a 10x run contains
- Common problems: empty droplets, ambient RNA, doublets and dropouts
- The steps of a standard scRNA-Seq analysis, which we will run in M8_L2 and M8_P1

## 1. Why single cell?

A bulk RNA-Seq experiment on a root answers the question "what is the average expression of each gene in this root?". Many biological questions are not about averages:

- **Which cell types express my gene?** A transcription factor detected at a low level in a bulk sample might be highly expressed in a handful of cells, such as the quiescent center.
- **Is a treatment effect general or cell-type specific?** In bulk data, a strong response in one rare cell type can be diluted to nothing, and a change in the *proportion* of cell types can look like a change in gene expression.
- **What are the steps of a developmental process?** Cells captured at different stages can be ordered along a trajectory ("pseudotime").

### Why the *Arabidopsis* root?
The *Arabidopsis* root tip is one of the best-mapped organs in plant biology, which makes it a natural test case for single-cell methods:

- Its tissues are arranged as **concentric cylinders**. From outside to inside they are the epidermis (made of root-hair cells, called *trichoblasts*, and non-hair cells, called *atrichoblasts*), the cortex, the endodermis, the pericycle and the vascular tissues (xylem and phloem). The root cap and columella sit at the tip.
- Development runs along the **length** of the root. Stem cells around the quiescent center (QC) divide in the meristem, and their daughters then elongate and differentiate as they move away from the tip. A single root therefore contains every stage of development at once.
- Many **marker genes** are already known from reporter lines and from bulk RNA-Seq of sorted cell populations, so the clusters found in a scRNA-Seq experiment can be checked against prior knowledge.

In this module we use the dataset of [Ryu *et al.* (2019)](https://doi.org/10.1104/pp.18.01482), which profiled about 7,500 wild-type root cells plus two epidermal mutants. It is described in detail in M8_L2.

## 2. Preparing plant cells: protoplasts or nuclei

Animal tissues can often be broken into single cells with gentle proteases. Plant cells are glued together by a **cell wall**, which has to be removed or bypassed before cells can be loaded onto a single-cell instrument.

| Approach | How it works | Advantages | Disadvantages |
|---|---|---|---|
| **Protoplasting** (single-cell RNA-Seq) | Tissue is incubated with cell-wall-degrading enzymes (cellulases and pectinases) to release wall-less cells called *protoplasts* | Captures cytoplasmic mRNA, so there are many transcripts per cell | Some cell types are harder to release than others, and the enzyme treatment itself induces a stress response |
| **Nuclei isolation** (single-nucleus RNA-Seq, snRNA-Seq) | Tissue is chopped or ground, and nuclei are purified by filtering or sorting | Works on almost any tissue (even frozen tissue), with less bias between cell types | Only nuclear RNA is captured, giving fewer transcripts per nucleus and more intronic reads |

```{note}
**Protoplasting-induced genes.** Digesting the cell wall takes one to two hours, and the cells respond to it by changing the expression of hundreds of genes. These changes were first described in sorted root cells by Birnbaum *et al.* (2003). Many plant scRNA-Seq studies therefore also sequence protoplasted and non-protoplasted bulk samples, and then flag or remove the genes that respond to the treatment. Denyer *et al.* (2019), for example, deposited such bulk samples in GEO (GSE123818). Keep this in mind when a "marker" turns out to be a stress or wounding gene.
```

## 3. scRNA-Seq platforms

There are three broad ways to keep the RNA of each cell separate:

| Platform family | Examples | How cells are separated | Typical throughput | Transcript coverage |
|---|---|---|---|---|
| **Plate-based** | Smart-seq2, CEL-Seq2 | One cell per well of a 96- or 384-well plate, usually placed there by FACS | Hundreds of cells | Smart-seq2 gives full-length transcripts, which is good for isoforms |
| **Droplet-based** | 10x Genomics Chromium, Drop-seq, inDrop | Each cell is enclosed in an oil droplet together with a barcoded bead | Thousands to tens of thousands of cells per run | 3' (or 5') end only |
| **Split-pool (combinatorial indexing)** | sci-RNA-seq, SPLiT-seq | Cells or nuclei are not isolated; instead, they are split into wells and barcoded over several rounds, and each cell ends up with a unique combination of barcodes | Up to hundreds of thousands to millions | 3' end only |

The most widely used platform for plants, and the one used by Ryu *et al.* (2019), is the **10x Genomics Chromium**, so the rest of this lab describes how it works. An *Arabidopsis* root dataset made with Drop-seq is also available (Shulse *et al.*, 2019); its chemistry is similar.

## 4. How a 10x Chromium 3' library is built

### 4.1 The gel bead
The key ingredient is the **gel bead**. Each bead carries millions of copies of a DNA oligonucleotide. All the oligos on one bead share the same **cell barcode**, and every oligo has its own random **UMI**. For the Single Cell 3' v2 chemistry used by Ryu *et al.*, each oligo reads (5' to 3'):

```
5'-[Illumina Read 1 primer site]-[Cell barcode, 16 nt]-[UMI, 10 nt]-[poly(dT), ~30 nt]VN-3'
```

For a base-by-base view of this oligo and of the final library molecule, see the Teichmann Lab's [single-cell library structure page for 10x Chromium 3'](https://teichlab.github.io/scg_lib_structs/methods_html/10xChromium3.html).

- **Read 1 primer site:** lets the sequencer read the barcode and UMI in Read 1.
- **Cell barcode (CB), 16 nt:** identical on every oligo of a given bead. There are ~737,000 known v2 barcodes (the *whitelist*), so two beads very rarely share one. **Same barcode = same droplet = (ideally) the same cell.**
- **Unique Molecular Identifier (UMI), 10 nt:** a random sequence that differs between oligos on the same bead. A UMI labels each individual mRNA molecule captured. The v3 chemistry uses a 12 nt UMI.
- **Poly(dT)VN:** binds the poly(A) tail of mRNA, so capture (and later sequencing) happens at the **3' end** of the transcript. The "VN" anchor makes the primer sit at the start of the poly(A) tail.

### 4.2 Making GEMs
The Chromium controller pushes three streams together in a microfluidic chip: the cells (or protoplasts) in buffer, the gel beads, and oil. The result is tens of thousands of nanolitre droplets called **GEMs** (Gel bead-in-EMulsion). Cells are loaded at a low concentration so that most GEMs contain a bead and **no** cell, and most of the remainder contain exactly **one** cell.

### 4.3 Barcoded reverse transcription inside each GEM
Inside a GEM, the bead dissolves and releases its oligos, the cell is lysed, and its poly(A) mRNA binds to the poly(dT) primers. Reverse transcription makes cDNA that now carries the cell barcode and a UMI. When it reaches the 5' end of the mRNA, the reverse transcriptase adds a few extra bases, and a **template switching oligo (TSO)** binds to them so that a known sequence is added at the other end of the cDNA. This makes it possible to amplify every cDNA with a single pair of primers.

### 4.4 Pooling and library construction
After RT, the emulsion is broken and **the cDNA from all cells is pooled into one tube**. From here on, the barcode is the only record of which cell each molecule came from.
1. **cDNA amplification:** PCR with primers against the Read 1 site and the TSO.
2. **Fragmentation, end repair and A-tailing:** the full-length cDNA is cut enzymatically into a few hundred bp. This is the same idea as the fragmentation step for bulk RNA-Seq in M2L2.
3. **Adapter ligation:** an adapter with the Illumina Read 2 primer site is ligated to the fragments.
4. **Sample index PCR:** adds the P5 and P7 flow cell sequences and an i7 **sample index**, so that several libraries can share one flow cell. Only fragments that still carry the bead end (Read 1 site + CB + UMI) *and* a Read 2 adapter are amplified. As a result, the library keeps the 3' ends of transcripts.

The finished library molecule looks like this:

```
[P5]-[Read 1 primer]-[CB 16 nt]-[UMI 10 nt]-[poly(dT)]-<---- cDNA insert (3' end of transcript) ---->-[Read 2 primer]-[i7 sample index]-[P7]
```

Compare this with the bulk RNA-Seq fragment from M2L2, `[P5] [Index 1] [Read 1 primer site] --- cDNA Fragment --- [Read 2 primer site] [Index 2] [P7]`. Here the cell barcode and UMI sit between the Read 1 primer and the cDNA, so they are read in Read 1.

## 5. Sequencing a 10x library: what is in each FASTQ?

10x libraries are sequenced on standard Illumina instruments, but the reads are **not** a normal paired-end pair:

| File | Typical length (3' v2) | Contains | Used for |
|---|---|---|---|
| `*_R1_001.fastq.gz` (Read 1) | 26 bp | 16 nt cell barcode + 10 nt UMI | Assigning reads to cells and collapsing PCR duplicates |
| `*_I1_001.fastq.gz` (Index 1) | 8 bp | i7 sample index | Demultiplexing samples (usually done by the sequencing core) |
| `*_R2_001.fastq.gz` (Read 2) | 98 bp or more | The cDNA insert | Aligning to the genome and assigning reads to genes |

```{note}
Only **Read 2** contains biological sequence and is aligned to the genome. Read 1 is treated as a *barcode read*: it is never aligned, and it should never be quality-trimmed, because cutting even one base would shift the barcode and UMI positions. This is the most important difference from the paired-end bulk RNA-Seq data you trimmed in M2P3 and M3P1.
```

Because only the 3' end of each transcript is sequenced, scRNA-Seq gives **gene-level counts**. It cannot tell isoforms apart, and it does not need fragment-length correction (TPM): one UMI corresponds to one captured molecule, whatever the length of the gene.

#### UMIs and PCR duplicates
In bulk RNA-Seq, two identical reads can come either from two different mRNA molecules or from PCR copies of one molecule, and we cannot tell which. In scRNA-Seq, reads that share the **same cell barcode, the same UMI and the same gene** are PCR copies of one molecule and are counted once. The final count matrix therefore holds **UMI counts**, not read counts.

```
CB               UMI          gene        -> counted as
AAACCTGAGCGATAGC ACGTTTGCAA   GL2         \
AAACCTGAGCGATAGC ACGTTTGCAA   GL2          > 1 molecule of GL2 in cell AAACCTGAGCGATAGC
AAACCTGAGCGATAGC ACGTTTGCAA   GL2         /
AAACCTGAGCGATAGC TTGACCAGTA   GL2         -> a 2nd molecule of GL2 in the same cell
GTCACAAGTCTAGTCA ACGTTTGCAA   GL2         -> a different cell (same UMI by chance)
```

#### Watch: an overview of single-cell sequencing
iBiology Techniques: *Single Cell Sequencing*, Eric Chow (UCSF). It covers why single-cell methods are used and how droplet barcoding works.

#SHIFT + ENTER TO SEE VIDEO
<iframe width="560" height="315" src="https://www.youtube.com/embed/k9VFNLLQP8c" title="Single Cell Sequencing - Eric Chow (UCSF)" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

## 6. What can go wrong: the main artifacts

| Artifact | Cause | What it looks like in the data | How it is handled |
|---|---|---|---|
| **Empty droplets** | Most GEMs contain a bead but no cell, yet still capture some free-floating RNA | Hundreds of thousands of barcodes with very few UMIs | *Cell calling*: Cell Ranger and STARsolo keep only barcodes that clearly stand out from the background (knee plot, EmptyDrops) |
| **Ambient RNA** | RNA released by broken cells (common after protoplasting) floats in the suspension and is captured in every droplet | Highly expressed genes from one cell type appear at low levels in all clusters | Tools such as SoupX or CellBender, or simply caution when reading markers |
| **Doublets / multiplets** | Two cells end up in one GEM | Barcodes with unusually many UMIs and genes, and mixed marker profiles | Load fewer cells; flag doublets with tools such as scDblFinder; set upper thresholds on genes per cell |
| **Damaged or dying cells** | Cells lyse during protoplasting or loading | Few genes, and a high fraction of organellar (mitochondrial or chloroplast) transcripts | QC filters on genes per cell and organellar % |
| **Dropouts (sparsity)** | Only a small fraction of each cell's mRNA is captured | Most entries in the count matrix are 0, even for expressed genes | Analyses work on many cells at once (clusters, smoothing) rather than trusting single zeros |

The doublet rate depends on how many cells are loaded. As a rule of thumb, 10x reports about 0.8 to 1% doublets per 1,000 cells recovered, so a run that recovers 5,000 cells has roughly 4 to 5% doublets.

```{note}
**Plants have two organellar genomes.** In animal scRNA-Seq, a high fraction of *mitochondrial* reads marks broken cells. In *Arabidopsis*, mitochondrial genes start with `ATMG` and chloroplast genes with `ATCG`. Roots have few chloroplasts, but plastid transcripts are still worth checking.
```

## 7. Designing a scRNA-Seq experiment

Many of the design questions from bulk RNA-Seq (M4L4, M6P2) still apply, with a few new ones:

- **Biological replicates are still essential.** Thousands of cells from *one* root sample are not replicates: they show cell-to-cell variation, not sample-to-sample variation. Ryu *et al.* sequenced three independent wild-type samples.
- **Cells versus depth.** With a fixed budget, you can sequence many cells shallowly or fewer cells deeply. To find cell types, more cells at moderate depth (for example 20,000 to 50,000 reads per cell) is usually better. To detect lowly expressed genes, depth matters more. Ryu *et al.* obtained about 75,000 reads per cell.
- **Rare cell types need many cells.** If the quiescent center makes up 0.1% of root cells, 5,000 cells will contain only about 5 QC cells.
- **Batch effects.** Samples processed on different days, protoplasted separately or run in different lanes can differ for technical reasons. They need to be integrated or batch-corrected during analysis (M8_P1).
- **Sample multiplexing** (hashtags or genotype-based demultiplexing) lets several samples share one 10x lane, which reduces both cost and batch effects.

## 8. The scRNA-Seq analysis workflow

The overall shape is the same as bulk RNA-Seq (raw reads → counts → statistics), but the matrix is now **genes × cells** instead of genes × samples:

| Step | What happens | Tools | Where in this module |
|---|---|---|---|
| 1. Read QC | Check read lengths, base quality and composition of R1 and R2 | FastQC, MultiQC, seqkit | M8_L2 |
| 2. Alignment and counting | Align R2 to the genome; correct cell barcodes against the whitelist; collapse UMIs; count UMIs per gene per cell; call cells | Cell Ranger, **STARsolo**, alevin-fry, kallisto\|bustools | M8_L2 |
| 3. Cell QC | Remove empty or damaged cells and likely doublets (genes per cell, UMIs per cell, organellar %) | Seurat, Scanpy | M8_P1 |
| 4. Normalization | Correct for differences in capture and sequencing depth between cells | `LogNormalize`, SCTransform | M8_P1 |
| 5. Feature selection | Keep the highly variable genes (HVGs) that drive differences between cells | `FindVariableFeatures` | M8_P1 |
| 6. Dimensionality reduction | Summarize thousands of HVGs as a few dozen principal components | PCA | M8_P1 |
| 7. Integration | Remove batch differences between replicates | CCA, RPCA, Harmony | M8_P1 |
| 8. Clustering | Group cells with similar profiles into clusters | Graph-based (Louvain or Leiden) | M8_P1 |
| 9. Visualization | Show cells in 2D | UMAP, t-SNE | M8_P1 |
| 10. Markers and annotation | Find the genes that distinguish each cluster, and name the clusters using known marker genes | `FindAllMarkers`, `FindConservedMarkers` | M8_P1 |
| 11. Downstream | Trajectories and pseudotime, differential expression between conditions | Monocle, Slingshot | (further reading) |

Cell Ranger is 10x Genomics' own pipeline, and Ryu *et al.* used it. In this course we use **STARsolo** (Kaminow *et al.*, 2021) instead. It is built into STAR, which you already used in M5L3, installs through conda, and was designed to give results very close to Cell Ranger's.

## Questions:
Answer the following questions in the text box below:
1. A bulk RNA-Seq experiment shows that gene *X* is 2-fold up-regulated in a mutant root. Give two different explanations for this result that only a single-cell experiment could tell apart.
2. Why is Read 1 of a 10x library never quality-trimmed or aligned to the genome? What would happen to the cell barcode and UMI if the first base of Read 1 were removed?
3. Three reads share the same cell barcode and UMI and map to the same gene. How many molecules do they represent, and why? What if they share a barcode and gene but have different UMIs?
4. Most droplets in a 10x run contain no cell. How can a pipeline tell an empty droplet from a real cell?
5. Give one advantage and one disadvantage of using protoplasts instead of nuclei for an *Arabidopsis* root experiment. How could you find out which genes in your data respond to protoplasting itself?
6. You can afford 200 million reads for one sample. Would you aim for 2,000 or 10,000 cells if your goal is to find rare cell types in the root tip? Explain your choice.

## References / Resources
- Ryu KH, Huang L, Kang HM, Schiefelbein J (2019). Single-cell RNA sequencing resolves molecular relationships among individual plant cells. *Plant Physiology* 179:1444-1456. [doi:10.1104/pp.18.01482](https://doi.org/10.1104/pp.18.01482) ([open access in PMC](https://pmc.ncbi.nlm.nih.gov/articles/PMC6446759/))
- Denyer T, Ma X, Klesen S, Scacchi E, Nieselt K, Timmermans MCP (2019). Spatiotemporal developmental trajectories in the *Arabidopsis* root revealed using high-throughput single-cell RNA sequencing. *Developmental Cell* 48:840-852. [doi:10.1016/j.devcel.2019.02.022](https://doi.org/10.1016/j.devcel.2019.02.022)
- Shulse CN, Cole BJ, Ciobanu D, *et al.* (2019). High-throughput single-cell transcriptome profiling of plant cell types. *Cell Reports* 27:2241-2247. [doi:10.1016/j.celrep.2019.04.054](https://doi.org/10.1016/j.celrep.2019.04.054)
- Birnbaum K, Shasha DE, Wang JY, *et al.* (2003). A gene expression map of the *Arabidopsis* root. *Science* 302:1956-1960. [doi:10.1126/science.1090022](https://doi.org/10.1126/science.1090022)
- Zheng GXY, Terry JM, Belgrader P, *et al.* (2017). Massively parallel digital transcriptional profiling of single cells. *Nature Communications* 8:14049. [doi:10.1038/ncomms14049](https://doi.org/10.1038/ncomms14049)
- Kaminow B, Yunusov D, Dobin A (2021). STARsolo: accurate, fast and versatile mapping/quantification of single-cell and single-nucleus RNA-seq data. *bioRxiv*. [doi:10.1101/2021.05.05.442755](https://doi.org/10.1101/2021.05.05.442755)
- HBC training: [Introduction to single-cell RNA-seq](https://hbctraining.github.io/Intro-to-scRNAseq/) (CC BY 4.0)
- Teichmann Lab: [Single-cell library structures, 10x Chromium 3'](https://teichlab.github.io/scg_lib_structs/methods_html/10xChromium3.html), a base-by-base view of the library molecule
- Galaxy Training Network: [Analysis of plant scRNA-Seq Data with Scanpy](https://training.galaxyproject.org/training-material/topics/single-cell/tutorials/scrna-plant/tutorial.html) (CC BY 4.0), a Python version of a root scRNA-Seq analysis